# Clase 07 — Tests de hipótesis I

**Curso:** Estadística Aplicada con Python · **Bloque 3:** Inferencia
**Tiempo estimado:** 120 minutos · **Capítulo del manual:** `manual/07_tests_hipotesis_i.md`

## Qué vas a ver en esta clase

- La **lógica del test**: H0 y H1, estadístico del test, *p-value*, nivel de significación y región de rechazo.
- La **interpretación correcta del p-value**, comprobada por simulación.
- **Errores de tipo I y II** y la **potencia** (a nivel conceptual): cómo cambia con el tamaño de la muestra.
- **z-test y t-test de una muestra**, **t-test de dos muestras independientes (Welch)**, **t-test pareado** y **test de dos proporciones**.
- El **tamaño del efecto** (diferencia de medias y d de Cohen) junto con el intervalo de confianza.
- La **comprobación de los supuestos** con gráficos.

## Requisitos

- Clase 1 (variabilidad de muestreo), clase 5 (distribución normal y *t*, teorema del límite central) y clase 6 (error estándar e intervalos de confianza).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Todos los datos (tiempos de carga, tasas de conversión) son sintéticos: un *ejemplo ilustrativo*, no datos reales.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla**: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. El problema: ¿la versión nueva de la página es más rápida?

Un equipo web lanza una versión nueva de una página. A 40 visitas se les sirve la versión antigua y a 45 la nueva (cada visita ve solo una). Medimos el **tiempo de carga** en segundos. La pregunta: ¿la diferencia que vemos en las medias es real o puede ser puro ruido de muestreo (clase 1)?

In [ ]:
rng_datos = np.random.default_rng(31)
antigua = rng_datos.normal(3.2, 0.9, 40).clip(0.8)      # ejemplo ilustrativo (s)
nueva   = rng_datos.normal(2.85, 0.9, 45).clip(0.8)

resumen = pd.DataFrame({
    "n": [len(antigua), len(nueva)],
    "media (s)": [antigua.mean(), nueva.mean()],
    "desv. típica (s)": [antigua.std(ddof=1), nueva.std(ddof=1)]},
    index=["versión antigua", "versión nueva"])
display(resumen.round(3))

plt.boxplot([antigua, nueva], tick_labels=["antigua", "nueva"])
plt.title("Tiempo de carga según la versión")
plt.ylabel("segundos")
plt.show()

---
## 2. La lógica del test: H0, estadístico y p-value

- **H0 (hipótesis nula):** no hay diferencia real; la media poblacional es la misma en las dos versiones.
- **H1 (hipótesis alternativa):** las medias son distintas.
- **Estadístico del test:** un número que resume cuánto se aleja lo observado de H0, medido en errores estándar.
- **p-value:** la probabilidad de obtener un resultado **al menos tan extremo** como el observado **si H0 fuera cierta**.

Calculamos a mano el estadístico t de Welch (diferencia / error estándar) y su p-value, y lo comparamos con `scipy`.

In [ ]:
n1, n2 = len(antigua), len(nueva)
dif = antigua.mean() - nueva.mean()
v1, v2 = antigua.var(ddof=1) / n1, nueva.var(ddof=1) / n2
error_estandar = np.sqrt(v1 + v2)
t_obs = dif / error_estandar
gl = (v1 + v2) ** 2 / (v1 ** 2 / (n1 - 1) + v2 ** 2 / (n2 - 1))     # grados de libertad de Welch
p_valor = 2 * stats.t.sf(abs(t_obs), gl)                              # dos colas

res = stats.ttest_ind(antigua, nueva, equal_var=False)
print(f"diferencia de medias = {dif:.3f} s   error estándar = {error_estandar:.3f}")
print(f"a mano:  t = {t_obs:.3f}   gl = {gl:.1f}   p = {p_valor:.4f}")
print(f"scipy:  t = {res.statistic:.3f}   gl = {res.df:.1f}   p = {res.pvalue:.4f}")
assert np.isclose(t_obs, res.statistic) and np.isclose(p_valor, res.pvalue)

Con **α = 0,05** (dos colas) la **región de rechazo** son los valores de t más extremos que el valor crítico: el 2,5 % de cada cola bajo H0. Si el t observado cae dentro, rechazamos H0.

In [ ]:
alfa = 0.05
t_crit = stats.t.ppf(1 - alfa / 2, gl)
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, gl)
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=(x <= -t_crit), color="tab:red", alpha=0.4, label="región de rechazo")
plt.fill_between(x, y, where=(x >= t_crit), color="tab:red", alpha=0.4)
plt.axvline(t_obs, color="tab:blue", linestyle="--", label=f"t observado = {t_obs:.2f}")
plt.title("Distribución de t bajo H0 y región de rechazo (α = 0,05)")
plt.xlabel("t"); plt.legend()
plt.show()
print(f"valor crítico = ±{t_crit:.3f}   →   ¿se rechaza H0?", abs(t_obs) > t_crit)

---
## 3. Una cola o dos colas

Si H1 es "la nueva es más rápida" (una dirección), el test es de **una cola**; si H1 es "son distintas" (cualquier dirección), de **dos colas**. Una cola concentra todo el α en un lado y da un p-value la mitad de grande **si el signo es el previsto**. Por eso la cola se elige **antes** de mirar los datos y según la pregunta de negocio; hacerlo después, viendo hacia dónde va la diferencia, es hacer trampa.

In [ ]:
for alt, etiqueta in [("two-sided", "dos colas (H1: distintas)"),
                      ("greater", "una cola (H1: antigua > nueva)"),
                      ("less", "una cola (H1: antigua < nueva)")]:
    r = stats.ttest_ind(antigua, nueva, equal_var=False, alternative=alt)
    print(f"{etiqueta:<45} p = {r.pvalue:.4f}")

**¿Qué observas?**

- El p-value de una cola en la dirección prevista es la mitad del de dos colas; en la dirección contraria es casi 1.
- Si hubiéramos decidido la cola *después* de ver que la versión nueva es más rápida, ¿qué p-value habríamos obtenido y qué rigor tendría?

*(Tu respuesta aquí.)*

---
## 4. Experimento guiado: qué significa realmente el p-value

Simulamos un mundo en el que **H0 es cierta**: las dos versiones tienen exactamente la misma media (3,0 s). Repetimos 10 000 veces el experimento (mismos tamaños, 40 y 45) y calculamos cada vez el test de Welch. Preguntas: ¿qué fracción de estos experimentos da un |t| al menos tan grande como el que observamos? ¿Con qué frecuencia sale "significativo" a α = 0,05?

In [ ]:
rng_sim = np.random.default_rng(101)
N_SIM = 10_000
g1 = rng_sim.normal(3.0, 0.9, (N_SIM, n1))
g2 = rng_sim.normal(3.0, 0.9, (N_SIM, n2))          # H0 cierta: misma media
sim = stats.ttest_ind(g1, g2, axis=1, equal_var=False)
p_sim = sim.pvalue

print(f"fracción con |t| ≥ |t observado| = {(np.abs(sim.statistic) >= abs(t_obs)).mean():.4f}   (p del test = {res.pvalue:.4f})")
print(f"fracción 'significativa' con α = 0,05 = {(p_sim < 0.05).mean():.4f}")
print(f"fracción 'significativa' con α = 0,01 = {(p_sim < 0.01).mean():.4f}")

plt.hist(p_sim, bins=20, edgecolor="white")
plt.axhline(N_SIM / 20, color="black", linestyle="--")
plt.title("p-values de 10 000 experimentos con H0 cierta")
plt.xlabel("p-value"); plt.ylabel("número de experimentos")
plt.show()

**¿Qué observas?**

- La fracción con |t| ≥ |t observado| es casi igual al p-value: es exactamente su definición ("¿con qué frecuencia vería algo así **si H0 fuera cierta**?").
- Con H0 cierta, aproximadamente el 5 % de los experimentos sale "significativo" a α = 0,05 y el 1 % a α = 0,01: **α es la tasa de falsos positivos** (error de tipo I) cuando H0 es cierta.
- El histograma es casi plano: bajo H0, todos los p-values son igual de probables. Un p = 0,03 no es "raro" si haces muchos tests (lo retomaremos en la clase 9).
- ¿Contesta este experimento a "¿cuál es la probabilidad de que H0 sea cierta?" ¿Por qué no?

*(Tu respuesta aquí.)*

---
## 5. Error de tipo II y potencia

| | H0 es cierta | H1 es cierta (hay efecto) |
|---|---|---|
| **Rechazo H0** | Error de tipo I (falso positivo), probabilidad α | Decisión correcta, probabilidad = **potencia** |
| **No rechazo H0** | Decisión correcta | Error de tipo II (falso negativo), probabilidad β = 1 − potencia |

La **potencia** es la probabilidad de detectar un efecto que **existe**. Depende del tamaño del efecto, del tamaño de la muestra, de la variabilidad y de α. Aquí solo miramos cómo cambia con n; el **cálculo del tamaño de muestra** se hace en la clase 13.

Simulamos un efecto real de 0,5 desviaciones típicas entre dos grupos y variamos el tamaño de cada grupo.

In [ ]:
rng_pot = np.random.default_rng(43)
N_SIM_P = 4_000
EFECTO_REAL = 0.5            # en desviaciones típicas (σ = 1)
filas = []
for n in [10, 20, 40, 80, 160]:
    x = rng_pot.normal(0, 1, (N_SIM_P, n))
    y = rng_pot.normal(EFECTO_REAL, 1, (N_SIM_P, n))
    potencia = (stats.ttest_ind(x, y, axis=1, equal_var=False).pvalue < 0.05).mean()
    filas.append({"n por grupo": n, "potencia (simulada)": potencia,
                  "prob. error tipo II": 1 - potencia})
tabla_potencia = pd.DataFrame(filas).set_index("n por grupo").round(3)
display(tabla_potencia)

plt.plot(tabla_potencia.index, tabla_potencia.iloc[:, 0], marker="o")
plt.axhline(0.8, color="black", linestyle="--")
plt.xscale("log")
plt.title("Potencia según el tamaño de cada grupo (efecto = 0,5 σ, α = 0,05)")
plt.xlabel("n por grupo (escala log)"); plt.ylabel("potencia")
plt.show()

**¿Qué observas?**

- Con 10 observaciones por grupo, ¿con qué frecuencia detectamos un efecto que **sí existe**? ¿Qué le pasa a un "no significativo" obtenido con una muestra pequeña?
- ¿Cuántas observaciones por grupo hacen falta, aproximadamente, para llegar al 80 % de potencia con este efecto?
- Si el efecto real fuera la mitad (0,25 σ), ¿la potencia con el mismo n sería mayor o menor?

*(Tu respuesta aquí.)*

---
## 6. Test de una muestra: z-test y t-test

Ocho mediciones del tiempo de una consulta (*ejemplo ilustrativo*). El objetivo del equipo es una media de 3,0 s. H0: μ = 3,0; H1: μ ≠ 3,0.

- Si la desviación típica poblacional **σ fuera conocida** (caso didáctico, casi nunca real) usaríamos el **z-test**: z = (x̄ − μ0) / (σ/√n), comparado con la normal.
- En la práctica σ se estima con s y se usa el **t-test**: t = (x̄ − μ0) / (s/√n), con n − 1 grados de libertad.

In [ ]:
muestra = np.array([3.4, 2.9, 3.8, 3.1, 3.6, 3.3, 3.9, 3.2])
mu0 = 3.0
n = len(muestra)
media, s = muestra.mean(), muestra.std(ddof=1)

t_mano = (media - mu0) / (s / np.sqrt(n))
p_t = 2 * stats.t.sf(abs(t_mano), n - 1)
r1 = stats.ttest_1samp(muestra, mu0)
print(f"media = {media:.3f}   s = {s:.4f}   error estándar = {s / np.sqrt(n):.4f}")
print(f"t-test a mano: t = {t_mano:.3f}   p = {p_t:.4f}   |   scipy: t = {r1.statistic:.3f}   p = {r1.pvalue:.4f}")

SIGMA_CONOCIDA = 0.35        # supuesto didáctico
z = (media - mu0) / (SIGMA_CONOCIDA / np.sqrt(n))
p_z = 2 * stats.norm.sf(abs(z))
print(f"z-test (σ = {SIGMA_CONOCIDA}): z = {z:.3f}   p = {p_z:.4f}")

Fíjate en que el valor de t con solo 7 grados de libertad exige más evidencia que el de z: la distribución t tiene colas más pesadas para compensar que estimamos σ con pocos datos (clase 5).

---
## 7. Dos muestras independientes: por qué Welch

El t-test clásico de Student supone **varianzas iguales** en los dos grupos. El de **Welch** (`equal_var=False`) no lo supone y ajusta los grados de libertad. Si las varianzas son parecidas, dan casi lo mismo; si no lo son (sobre todo con tamaños distintos), Student puede equivocarse mucho.

Experimento: H0 cierta (misma media), pero un grupo pequeño (n = 10) con mucha variabilidad (σ = 3) y otro grande (n = 50) con poca (σ = 1). ¿Qué tasa de falsos positivos da cada test?

In [ ]:
rng_w = np.random.default_rng(202)
a = rng_w.normal(0, 3, (10_000, 10))
b = rng_w.normal(0, 1, (10_000, 50))          # H0 cierta: ambas con media 0
fp_student = (stats.ttest_ind(a, b, axis=1, equal_var=True).pvalue < 0.05).mean()
fp_welch = (stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue < 0.05).mean()
print(f"Falsos positivos (α = 0,05):  Student = {fp_student:.3f}   Welch = {fp_welch:.3f}")

r_student = stats.ttest_ind(antigua, nueva, equal_var=True)
print(f"Con los datos de la web (varianzas parecidas):  Student p = {r_student.pvalue:.4f}   Welch p = {res.pvalue:.4f}")

**¿Qué observas?**

- Student, con varianzas y tamaños muy distintos, da muchos más "significativos" de los que promete α (aquí más de 5 veces el 5 %). Welch se queda cerca del 5 %.
- Con los datos de la web la diferencia entre ambos es mínima. Entonces, ¿qué perdemos usando siempre Welch? (Casi nada: por eso es la elección prudente.)

*(Tu respuesta aquí.)*

---
## 8. Muestras pareadas

Ahora medimos las **mismas 25 páginas** antes y después de la optimización. Las páginas difieren mucho entre sí (unas son ligeras y otras pesadas), pero cada una se compara **consigo misma**. El t-test pareado (`ttest_rel`) trabaja sobre las **diferencias** por página, y así elimina la variabilidad entre páginas.

In [ ]:
rng_p = np.random.default_rng(5)
antes = rng_p.lognormal(1.1, 0.5, 25)                          # ejemplo ilustrativo (s)
despues = antes - 0.20 + rng_p.normal(0, 0.25, 25)
difs = antes - despues

r_par = stats.ttest_rel(antes, despues)
r_ind = stats.ttest_ind(antes, despues, equal_var=False)       # INCORRECTO para datos pareados
r_dif = stats.ttest_1samp(difs, 0)
print(f"diferencia media por página = {difs.mean():.3f} s   desv. típica de las diferencias = {difs.std(ddof=1):.3f}")
print(f"Pareado:    t = {r_par.statistic:.3f}   p = {r_par.pvalue:.2e}")
print(f"Independiente (mal): t = {r_ind.statistic:.3f}   p = {r_ind.pvalue:.4f}")
print(f"Una muestra sobre las diferencias: t = {r_dif.statistic:.3f}   p = {r_dif.pvalue:.2e}")

for i in range(25):
    plt.plot([0, 1], [antes[i], despues[i]], color="gray", alpha=0.6, marker="o")
plt.xticks([0, 1], ["antes", "después"])
plt.title("Tiempo de carga de las mismas 25 páginas")
plt.ylabel("segundos")
plt.show()

**¿Qué observas?**

- Casi todas las líneas bajan, pero las páginas parten de niveles muy distintos. ¿Por qué el test independiente no ve el efecto y el pareado sí?
- El t-test pareado es idéntico a un t-test de una muestra sobre las diferencias (mismo t y p).
- ¿Cuándo **no** se puede usar el pareado? (Cuando los grupos no tienen una correspondencia uno a uno, como en la sección 1, donde cada visita ve una sola versión.)

*(Tu respuesta aquí.)*

---
## 9. Dos proporciones (tasas de conversión)

Otra métrica de la página: la **tasa de conversión** (visitas que compran). Grupo antiguo: 200 conversiones de 4 000 visitas (5,0 %); grupo nuevo: 244 de 4 000 (6,1 %) (*ejemplo ilustrativo*). H0: las dos proporciones son iguales. Este es el test que usarás en el A/B (clase 13).

`proportions_ztest` usa una desviación típica **combinada** (bajo H0 las dos proporciones son iguales, así que se juntan los datos). El intervalo de confianza de la diferencia usa, en cambio, el error estándar **sin combinar**.

In [ ]:
conversiones = np.array([244, 200])          # nueva, antigua
visitas = np.array([4000, 4000])
z_prop, p_prop = proportions_ztest(conversiones, visitas)

p_nueva, p_antigua = conversiones / visitas
p_comb = conversiones.sum() / visitas.sum()
ee_comb = np.sqrt(p_comb * (1 - p_comb) * (1 / 4000 + 1 / 4000))
z_mano = (p_nueva - p_antigua) / ee_comb

ee_sin = np.sqrt(p_nueva * (1 - p_nueva) / 4000 + p_antigua * (1 - p_antigua) / 4000)
ic_dif = ((p_nueva - p_antigua) - 1.96 * ee_sin, (p_nueva - p_antigua) + 1.96 * ee_sin)

print(f"diferencia de proporciones = {p_nueva - p_antigua:.4f}   IC 95 % = ({ic_dif[0]:.4f}, {ic_dif[1]:.4f})")
print(f"statsmodels: z = {z_prop:.3f}   p = {p_prop:.4f}   |   a mano: z = {z_mano:.3f}")

# mismas tasas aproximadas con solo 400 visitas por grupo
z_peq, p_peq = proportions_ztest(np.array([26, 20]), np.array([400, 400]))
print(f"Con 400 visitas por grupo (6,5 % vs 5,0 %): z = {z_peq:.3f}   p = {p_peq:.4f}")

**¿Qué observas?**

- Con 4 000 visitas por grupo, la diferencia de 1,1 puntos porcentuales sale significativa, pero el IC llega casi hasta 0. ¿Qué te dice sobre la precisión de la estimación?
- Con solo 400 visitas por grupo, una diferencia mayor (1,5 puntos) **no** es significativa. ¿Significa que no hay efecto? (No: significa que no hay evidencia suficiente con esa muestra.)

*(Tu respuesta aquí.)*

---
## 10. Tamaño del efecto: significativo no es lo mismo que importante

El p-value dice si el efecto es **compatible con el azar**, no si es **grande**. Se informan siempre:

- la **diferencia de medias** en unidades originales (segundos), con su **intervalo de confianza** (clase 6);
- la **d de Cohen**: la diferencia de medias dividida por la desviación típica combinada, es decir, la diferencia en "desviaciones típicas". Se lee siempre **en su contexto**: 0,2 s pueden ser irrelevantes en una página informativa y críticos en un pago.

Aplicamos ambas cosas a los datos web y, después, a un caso con n enorme.

In [ ]:
def d_de_cohen(a, b):
    """Diferencia de medias / desviación típica combinada."""
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / sp

ic = res.confidence_interval(confidence_level=0.95)
print(f"Datos web: diferencia = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d_de_cohen(antigua, nueva):.2f}   p = {res.pvalue:.4f}")

rng_g = np.random.default_rng(303)
grande_a = rng_g.normal(3.0, 0.9, 20_000)
grande_b = rng_g.normal(2.96, 0.9, 20_000)
r_g = stats.ttest_ind(grande_a, grande_b, equal_var=False)
ic_g = r_g.confidence_interval(confidence_level=0.95)
print(f"n = 20 000 por grupo: diferencia = {grande_a.mean() - grande_b.mean():.3f} s   IC 95 % = ({ic_g.low:.3f}, {ic_g.high:.3f})   d = {d_de_cohen(grande_a, grande_b):.2f}   p = {r_g.pvalue:.1e}")

**¿Qué observas?**

- El segundo caso es "muy significativo" (p minúsculo) pero la mejora es de unas 0,04 s, con d ≈ 0,04. ¿Justifica ese cambio el coste de desplegarlo? Eso depende del negocio, no del p-value.
- En los datos web, d ≈ 0,5 y una mejora de unos 0,4 s: más relevante, pero con un IC bastante ancho (de unos 0,05 s a unos 0,75 s).

*(Tu respuesta aquí.)*

---
## 11. Comprobar los supuestos

Los t-tests suponen que los datos son aproximadamente normales (o que n es suficientemente grande para que la media lo sea, clase 5), observaciones independientes y, en el caso de Student, varianzas iguales. Lo primero es **mirar**: histograma y gráfico Q-Q. Un test de normalidad (Shapiro-Wilk) es un complemento con **límites**: con n pequeño casi nunca rechaza (poca potencia) y con n grande rechaza desviaciones sin importancia práctica.

In [ ]:
rng_n = np.random.default_rng(404)
asimetrica = rng_n.lognormal(0.9, 0.6, 30)        # ejemplo ilustrativo: tiempos con cola larga

fig, ejes = plt.subplots(2, 2, figsize=(9, 6))
for fila, (datos, nombre) in enumerate([(nueva, "versión nueva (aprox. normal)"),
                                        (asimetrica, "muestra asimétrica")]):
    ejes[fila, 0].hist(datos, bins=10, edgecolor="white")
    ejes[fila, 0].set_title(nombre)
    stats.probplot(datos, dist="norm", plot=ejes[fila, 1])
    ejes[fila, 1].set_title("Gráfico Q-Q")
plt.tight_layout()
plt.show()

print(f"Shapiro-Wilk versión nueva: p = {stats.shapiro(nueva).pvalue:.3f}")
print(f"Shapiro-Wilk muestra asimétrica (n = 30): p = {stats.shapiro(asimetrica).pvalue:.3f}   asimetría = {stats.skew(asimetrica):.2f}")

rng_t = np.random.default_rng(505)
casi_normal = rng_t.standard_t(10, 5_000)
print(f"Shapiro-Wilk datos casi normales pero n = 5 000: p = {stats.shapiro(casi_normal).pvalue:.1e}")

**¿Qué observas?**

- La muestra asimétrica se ve claramente sesgada en el histograma y el Q-Q, pero con n = 30 Shapiro-Wilk **no** rechaza la normalidad (p ≈ 0,10): el test tiene poca potencia con n pequeño.
- Con n = 5 000 y datos casi normales, Shapiro-Wilk rechaza con un p minúsculo: con muestras grandes detecta desviaciones irrelevantes para un t-test.
- Moraleja: decide con los gráficos y el sentido común; si los datos son muy asimétricos o hay valores extremos y n es pequeño, la clase 8 presenta los tests no paramétricos.

*(Tu respuesta aquí.)*

---
## Resumen de la clase

- Un test compara lo observado con lo que esperaríamos **si H0 fuera cierta**; el **p-value** es la probabilidad de un resultado al menos así de extremo bajo H0. **No** es la probabilidad de que H0 sea cierta ni de haberse equivocado.
- **α** es la tasa de falsos positivos con H0 cierta (lo comprobamos: ≈ 5 % con α = 0,05). La **potencia** es la probabilidad de detectar un efecto real y crece con n.
- Una cola o dos se decide **antes** de ver los datos.
- Herramientas: `ttest_1samp`, `ttest_ind(equal_var=False)` (Welch), `ttest_rel` (solo si hay pareja uno a uno) y `proportions_ztest`.
- Informa siempre **efecto + IC + p-value**. "No significativo" no es "no hay efecto"; "significativo" no es "importante".
- Cada test extra que pruebas hasta encontrar p < 0,05 infla los falsos positivos: la clase 9 lo aborda (comparaciones múltiples).

**Siguiente paso:** el notebook de práctica de la clase 7 y, después, la clase 8 (ANOVA, tests no paramétricos) y la clase 9 (chi-cuadrado, comparaciones múltiples).